<a href="https://colab.research.google.com/github/AmitabhDey-byte/ANN/blob/main/ANN_on_mnist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [83]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(device)

cuda


In [84]:
train_df = pd.read_csv("fashion-mnist_train.csv")
test_df = pd.read_csv("fashion-mnist_test.csv")

In [85]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60000 entries, 0 to 59999
Columns: 785 entries, label to pixel784
dtypes: int64(785)
memory usage: 359.3 MB


In [86]:
class MNISTDataset(Dataset):

    def __init__(self, dataframe):
        self.X = dataframe.iloc[:, 1:].values.astype("float32") / 255.0
        self.y = dataframe.iloc[:, 0].values.astype("int64")

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        image = torch.tensor(self.X[idx], dtype=torch.float32)
        label = torch.tensor(self.y[idx], dtype=torch.long)

        return image, label

In [87]:
train_dataset = MNISTDataset(train_df)
test_dataset = MNISTDataset(test_df)

In [88]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

In [89]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)

torch.Size([64, 784])
torch.Size([64])


In [90]:
from torch.nn.modules.linear import Linear
class MyNN(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(num_features, 512),
            nn.GELU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.model(x)

In [91]:
model = MyNN(train_df.shape[1] - 1).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.002)

In [92]:
epochs = 120

In [93]:
for epochs in range(epochs):
    total_epoch_loss = 0  # Initialize total_epoch_loss for each epoch
    for batch_idx, (data, targets) in enumerate(train_loader):
        data = data.to(device)
        targets = targets.to(device)
        output = model(data)
        loss = criterion(output, targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_epoch_loss += loss.item()

    avg_loss = total_epoch_loss / len(train_loader)
    print(f"Epoch: {epochs+1}, Loss: {avg_loss}")

Epoch: 1, Loss: 0.5006406036044743
Epoch: 2, Loss: 0.370139888966325
Epoch: 3, Loss: 0.3293857277551694
Epoch: 4, Loss: 0.3081058298370668
Epoch: 5, Loss: 0.28914811995539713
Epoch: 6, Loss: 0.27416475979027466
Epoch: 7, Loss: 0.2651926898625868
Epoch: 8, Loss: 0.2548772367650766
Epoch: 9, Loss: 0.245441788676451
Epoch: 10, Loss: 0.2367627091689913
Epoch: 11, Loss: 0.22785814903946575
Epoch: 12, Loss: 0.2197962083152807
Epoch: 13, Loss: 0.21612746987356815
Epoch: 14, Loss: 0.21231751127251938
Epoch: 15, Loss: 0.20362428674247982
Epoch: 16, Loss: 0.1983651726333889
Epoch: 17, Loss: 0.1973482006982064
Epoch: 18, Loss: 0.18748426302743237
Epoch: 19, Loss: 0.18511364624492013
Epoch: 20, Loss: 0.18326318957038654
Epoch: 21, Loss: 0.17705230949831796
Epoch: 22, Loss: 0.17256247455567947
Epoch: 23, Loss: 0.169720706357154
Epoch: 24, Loss: 0.16833822285808098
Epoch: 25, Loss: 0.16414481047779195
Epoch: 26, Loss: 0.1621536800204945
Epoch: 27, Loss: 0.15920815775905656
Epoch: 28, Loss: 0.1530643

In [94]:
model.eval()

MyNN(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=512, out_features=256, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): ReLU()
    (6): Linear(in_features=128, out_features=10, bias=True)
  )
)

In [99]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = accuracy_score(all_labels, all_preds)*100
precision = precision_score(all_labels, all_preds, average="macro")*100
recall = recall_score(all_labels, all_preds, average="macro")*100
f1 = f1_score(all_labels, all_preds, average="macro")*100

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

Accuracy : 90.1900
Precision: 90.2835
Recall   : 90.1900
F1 Score : 90.2213
